In [1]:
from tad_mctc import units

onebohr = 1
print(f"{onebohr} bohr is equal to {onebohr*units.AU2AA} Angstrom.")

1 bohr is equal to 0.529177210903 Angstrom.


In [8]:
import torch
import dxtb
from dxtb import timer
from tad_mctc.units import AA2AU

timer.disable()
dd = {"dtype": torch.double, "device": torch.device("cpu")}

# C12H11ClNO
numbers = torch.tensor([1, 6, 7, 6, 6, 6, 6, 6, 6, 6, 17, 6, 6, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], device=dd["device"])
positions = torch.tensor([[-1.1114834526, 1.6094932194, -1.9331164053], [-1.5420758704, 0.9264053946, -1.0295958406], [-2.7998947492, 0.3901707568, -0.9499828031], [-3.0798225155, -0.3450547044, 0.2701855393], [-1.6644553062, -0.6583011355, 0.7908986734], [-0.8085545958, 0.5176051925, 0.2691921396], [0.6697544643, 0.2462745422, 0.1225187619], [1.1817921402, -0.4470233574, -0.9860740901], [2.5424282061, -0.7382418619, -1.0923697091], [3.4130793875, -0.333418246, -0.076572089], [5.1208508658, -0.6952288215, -0.2025351142], [2.9327020409, 0.3627707447, 1.0345976298], [1.5679371316, 0.6468035778, 1.1226404623], [-3.5048404036, 0.6422995872, -1.6337841006], [-3.651866459, 0.2657274597, 0.9958601638], [-3.6661706337, -1.2563590389, 0.07023373], [-1.3017637484, -1.596493753, 0.3412719265], [-1.6275282242, -0.7734336107, 1.8835060808], [-0.9357233997, 1.3802088926, 0.949278011], [0.5135106936, -0.7512109932, -1.7948906059], [2.9327126095, -1.271115929, -1.9610179711], [3.6227349699, 0.6811371164, 1.8177653429], [1.1967768489, 1.1972849675, 1.9919902676]], **dd)
positions = positions * AA2AU

# instantiate a calculator
calc = dxtb.calculators.GFN1Calculator(numbers, **dd)

# compute the energy
pos = positions.clone().requires_grad_(True)
energy = calc.get_energy(pos)

# obtain gradient (dE/dR) via autograd
(g,) = torch.autograd.grad(energy, pos)

# Alternatively, forces can directly be requested from the calculator.
# (Don't forget to manually reset the calculator when the inputs are identical.)
calc.reset()
pos = positions.clone().requires_grad_(True)
forces = calc.get_forces(pos)

assert torch.equal(forces, -g)

Total Energy: -34.43711594389217 Hartree.
Total Energy: -34.43711594389217 Hartree.


In [9]:
forces

tensor([[-0.0256, -0.0370,  0.0469],
        [ 0.0463,  0.0230, -0.0160],
        [-0.0209, -0.0019, -0.0172],
        [ 0.0037,  0.0005,  0.0011],
        [ 0.0023, -0.0034, -0.0024],
        [-0.0064,  0.0204, -0.0175],
        [ 0.0068, -0.0081, -0.0007],
        [ 0.0018,  0.0012,  0.0011],
        [-0.0011,  0.0033,  0.0042],
        [ 0.0015, -0.0001,  0.0002],
        [-0.0115,  0.0024,  0.0009],
        [-0.0030, -0.0016, -0.0054],
        [ 0.0014, -0.0002,  0.0002],
        [ 0.0006, -0.0033,  0.0079],
        [ 0.0031, -0.0006, -0.0036],
        [ 0.0031,  0.0036, -0.0014],
        [-0.0018,  0.0035,  0.0016],
        [-0.0005,  0.0031, -0.0052],
        [ 0.0032, -0.0038,  0.0045],
        [ 0.0016,  0.0011,  0.0044],
        [-0.0025,  0.0025,  0.0044],
        [-0.0040, -0.0015, -0.0037],
        [ 0.0018, -0.0032, -0.0043]], dtype=torch.float64)